# ETL Pipeline

In [1]:
import pandas as pd

### Load cleaned CSV

In [2]:
uof = pd.read_csv("cleaned_data.csv")

In [3]:
uof.head

<bound method NDFrame.head of                         ID           Incident_Type            Date_Time  \
0   2015UOF-0072-1204-3004  Level 2 - Use of Force  2015-01-19 15:00:00   
1   2015UOF-1321-1306-5300  Level 2 - Use of Force  2015-08-03 15:30:00   
2  2020UOF-1310-1097-23017  Level 2 - Use of Force  2020-07-19 15:35:00   
3  2022UOF-1269-2902-29489  Level 2 - Use of Force  2022-11-10 06:29:00   

   Officer_ID  Subject_ID                   Subject_Race Subject_Gender  Year  \
0        1634        2984                          White           Male  2015   
1        1615        5261                          White           Male  2015   
2        1672       23898  Nat Hawaiian/Oth Pac Islander           Male  2020   
3        5708       30362                          White         Female  2022   

   Month  
0      1  
1      8  
2      7  
3     11  >

### Load external dataset

In [5]:
encamp = pd.read_csv("Unauthorized_Encampment_Reports_20260221.csv")

/var/folders/nf/xj9w2p992vzfzsk5h40k9hb40000gn/T/ipykernel_60612/1494922539.py:1: DtypeWarning: Columns (10,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  encamp = pd.read_csv("Unauthorized_Encampment_Reports_20260221.csv")


In [6]:
encamp.head

<bound method NDFrame.head of        Service Request Number            Created Date     Method Received   \
0                 18-00062016  04/03/2018 11:49:19 AM                Phone   
1                 18-00062201  04/03/2018 02:28:10 PM                Phone   
2                 18-00062281  04/03/2018 03:34:14 PM                Phone   
3                 18-00062298  04/03/2018 03:45:00 PM  Find It Fix It Apps   
4                 18-00062310  04/03/2018 03:54:26 PM                Phone   
...                       ...                     ...                  ...   
256736            26-00048662  02/20/2026 04:37:50 PM  Find It Fix It Apps   
256737            26-00048682  02/20/2026 05:06:03 PM  Find It Fix It Apps   
256738            26-00048725  02/20/2026 06:24:39 PM  Find It Fix It Apps   
256739            26-00048746  02/20/2026 07:45:16 PM  Find It Fix It Apps   
256740            26-00048797  02/21/2026 01:25:52 AM  Find It Fix It Apps   

        Status                   

In [7]:
print("Use of Force rows/cols:", uof.shape)

Use of Force rows/cols: (4, 9)


In [8]:
print("Encampment rows/cols:", encamp.shape)

Encampment rows/cols: (256741, 19)


In [9]:
print("\nUse of Force columns:\n", uof.columns)


Use of Force columns:
 Index(['ID', 'Incident_Type', 'Date_Time', 'Officer_ID', 'Subject_ID',
       'Subject_Race', 'Subject_Gender', 'Year', 'Month'],
      dtype='object')


In [10]:
print("\nEncampment columns:\n", encamp.columns)


Encampment columns:
 Index(['Service Request Number', 'Created Date', 'Method Received ', 'Status',
       'Location', 'X_Value', 'Y_Value', 'Latitude', 'Longitude',
       'Latitude/Longitude', 'ZIP Code', 'Council District', 'Police Precinct',
       'Community Reporting Area', 'Are there people present?',
       'Are there tents, structures, or tarps?',
       'Are there RVs/cars/misc. vehicles?',
       'Is the encampment blocking access?', 'Is there trash or debris?'],
      dtype='object')


### Encampment data origin

This encampment data is from the City of Seattle website: https://data.seattle.gov/City-Administration/Unauthorized-Encampment-Reports/k7ra-jqqe/about_data

The data were pulled on Sat, Feb 21

### Why use encampment data?

Encampment data were chosen to enrich the Use of Force data with community and environment context.

Encampment reports can be joined/compared by geography, and or date/time data, enabling analysis of whether incident patterns align with encampment reporting patterns.

# Transform encampment data

In [11]:
# fix column names
encamp.columns = (
    encamp.columns
    .str.strip()
    .str.lower()
    .str.replace("/", "_", regex=False)
    .str.replace("?", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

In [13]:
encamp.columns

Index(['service_request_number', 'created_date', 'method_received', 'status',
       'location', 'x_value', 'y_value', 'latitude', 'longitude',
       'latitude_longitude', 'zip_code', 'council_district', 'police_precinct',
       'community_reporting_area', 'are_there_people_present',
       'are_there_tents,_structures,_or_tarps',
       'are_there_rvs_cars_misc_vehicles', 'is_the_encampment_blocking_access',
       'is_there_trash_or_debris'],
      dtype='object')

In [ ]:
# fix datatype - created_date to daatetime format
encamp["created_date"] = pd.to_datetime(encamp["created_date"], errors="coerce")

In [15]:
# drop columns and keep ones we'll use
keep_cols = [
    "service_request_number",
    "created_date",
    "status",
    "police_precinct",
    "community_reporting_area",
    "council_district",
    "zip_code",
    "latitude",
    "longitude",
    "are_there_people_present",
    "are_there_tents_structures_or_tarps",
    "are_there_rvs_cars_misc_vehicles",
    "is_the_encampment_blocking_access",
    "is_there_trash_or_debris",
]
encamp = encamp[[c for c in keep_cols if c in encamp.columns]].copy()


In [16]:
encamp.head

<bound method NDFrame.head of        service_request_number        created_date  status police_precinct  \
0                 18-00062016 2018-04-03 11:49:19  Closed           NORTH   
1                 18-00062201 2018-04-03 14:28:10  Closed           SOUTH   
2                 18-00062281 2018-04-03 15:34:14  Closed           NORTH   
3                 18-00062298 2018-04-03 15:45:00  Closed            EAST   
4                 18-00062310 2018-04-03 15:54:26  Closed            WEST   
...                       ...                 ...     ...             ...   
256736            26-00048662 2026-02-20 16:37:50  Closed           NORTH   
256737            26-00048682 2026-02-20 17:06:03    Open       SOUTHWEST   
256738            26-00048725 2026-02-20 18:24:39    Open           NORTH   
256739            26-00048746 2026-02-20 19:45:16    Open           NORTH   
256740            26-00048797 2026-02-21 01:25:52    Open           NORTH   

         community_reporting_area  council_di